# IterativeFib (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/IterativeFib_cpp.ipynb)

Chapter 14 — Recursion. Runnable notebook of `IterativeFib`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/IterativeFib_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/IterativeFib_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are global variables, and the trace is switched on with `trace = true`.

## Iterative Fibonacci: IterativeFib

**Input:** a natural number $n$. **Output:** $F(n)$, computed from $F(0)$ and $F(1)$ upwards keeping only the last two values. $S(n) = 5n + 1 = \Theta(n)$ steps. Invariant: before every evaluation of the condition, $prev = F(i-2)$ and $curr = F(i-1)$.

Pseudo-code: section 14.5.2 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [1]:
%%writefile iterative_fib.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = number of executions of line r (numbering of the lecture notes)
long long calls = 0;    // number of calls = nodes of the tree of recursive calls
int maxDepth = 0;       // depth of the recursion = maximum height of the stack
long long mults = 0;    // multiplications (FastPower)
long long moves = 0;    // moves (Hanoi)
int ev = 0;             // counter of the events of the trace
int stk[1000];          // the call stack: the arguments of the active calls (at most 1000)
int sp = 0;             // number of active calls
long long seen[64];     // seen[x] = number of calls with argument x
bool trace = false;     // true: print the trace of the execution

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Line | Executions\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Total steps S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // like sprintf, but returns a string
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // a call starts: its frame goes on top of the stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // the call returns: its frame leaves the stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // numbered line of the trace, with the stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // line of the trace: the call, indented one level for every active call
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

// IterativeFib(n)
long long IterativeFib(int n, bool trace = false)
{
    cnt[2]++;
    if (n <= 1)
    {
        cnt[3]++;
        return n;
    }
    long long prev = 0; cnt[5]++;
    long long curr = 1; cnt[6]++;
    int i = 2; cnt[7]++;
    if (trace)                                     // trace: initialization row
        printf("%-10s | %-8s | %-18s | %4lld | %4lld\n", "start", "2", "-", prev, curr);
    cnt[8]++;
    while (i <= n)
    {
        long long next = prev + curr; cnt[9]++;
        prev = curr; cnt[10]++;
        curr = next; cnt[11]++;
        i = i + 1; cnt[12]++;
        cnt[8]++;
        if (trace)                                 // trace: i - 2 is the number of the iteration
            printf("%-10d | %-8s | %-18s | %4lld | %4lld\n", i - 2, Str("%d -> %d", i - 1, i).c_str(), Str("%lld + %lld = %lld", next - prev, prev, next).c_str(), prev, curr);
    }
    cnt[14]++;
    return curr;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Example of the lecture notes: IterativeFib(4)
    int n = 4;
    Reset();
    printf("iteration | i        | next = prev + curr | prev | curr\n");
    long long r = IterativeFib(n, true);
    printf("The loop ends with i = %d > %d: it returns curr = %lld\n", n + 1, n, r);
    PrintSteps({2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14});
    printf("S(n) = 5n + 1 = %d\n", 5 * n + 1);

    // Comparison with Fib: steps of Fib = 4F(n+1) - 2 (formula checked in the previous section)
    printf("\n");
    printf("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1\n");
    int values[] = {5, 10, 20, 30};
    for (int v : values)
    {
        Reset();
        r = IterativeFib(v);
        long long steps = Total();
        printf("%3d | %6lld | %16lld | %12lld | %d\n", v, r, 4 * IterativeFib(v + 1) - 2, steps, 5 * v + 1);
    }
    return 0;
}
#endif

Writing iterative_fib.cpp


In [2]:
!g++ -std=c++17 -O2 -o iterative_fib iterative_fib.cpp && ./iterative_fib

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
The loop ends with i = 5 > 4: it returns curr = 3
Line | Executions
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Total steps S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Try it yourself.** Change $n$ and check that the steps are $5n + 1$.

In [3]:
%%writefile prova_iterative_fib.cpp
#define SOLO_LIBRERIA   // do not run the main of iterative_fib.cpp
#include "iterative_fib.cpp"

int main()
{
    int n = 10;        // <-- change n
    Reset();
    printf("iteration | i        | next = prev + curr | prev | curr\n");
    long long r = IterativeFib(n, true);
    printf("F(%d) = %lld, S = %lld = 5n + 1 = %d\n", n, r, Total(), 5 * n + 1);
    return 0;
}

Writing prova_iterative_fib.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_iterative_fib prova_iterative_fib.cpp && ./prova_iterative_fib

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).